In [1]:
## IMPORTS 
import evotoon
from data_classes import CatParam, IntParam, FloatParam


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
#import seaborn as sns#

import random

In [2]:
## MAKE SEED
SEED = evotoon.make_seed(283)
separator = "--------------------------------------------------------"

In [3]:
## PROTOTYPE EXAMPLE WITH AntKnapsackClean-Master

# Env configuration
poblation_size = 10

# Parameter settings
float_params = [
	FloatParam("tau_max", 0.02, 4.0),
	FloatParam("rho", 0.001, 1.0),
	FloatParam("alpha", 1.0, 8.0),
	FloatParam("beta", 1.0, 8.0),	
]
int_params = [
	IntParam("total_ants", 2, 25),
]

initial_batch = evotoon.initialization(poblation_size, float_params, int_params)

# SET ENVIRONMENT FOR THE ILSMKP ALGORITHM TO TUNE
instance_list = [
	"./AntKnapsackClean-master/instances/weing1.txt",
	"./AntKnapsackClean-master/instances/weing2.txt",
	"./AntKnapsackClean-master/instances/weing3.txt",
	"./AntKnapsackClean-master/instances/weing4.txt",
	"./AntKnapsackClean-master/instances/weing7.txt",
]

seed_list = [SEED + i for i in range(len(instance_list))]
print(separator, "Instances running and seeds", separator)
for ins,seed in zip(instance_list,seed_list):
	print("instance:", ins, "seed:", seed)

function_kwargs = {
	"executable_path": "./AntKnapsackClean-master/AntKnapsack",
	"instance_list": instance_list,
	"seed_list": seed_list,
	"evaluations": 25,
	"tau_min": 0.01
}

print(pd.DataFrame(initial_batch))

-------------------------------------------------------- Instances running and seeds --------------------------------------------------------
instance: ./AntKnapsackClean-master/instances/weing1.txt seed: 283
instance: ./AntKnapsackClean-master/instances/weing2.txt seed: 284
instance: ./AntKnapsackClean-master/instances/weing3.txt seed: 285
instance: ./AntKnapsackClean-master/instances/weing4.txt seed: 286
instance: ./AntKnapsackClean-master/instances/weing7.txt seed: 287
    tau_max       rho     alpha      beta  total_ants
0  1.071048  0.371641  6.874519  6.688981          17
1  3.838395  0.982210  3.343147  1.353605          24
2  1.282402  0.069259  4.465537  6.081290           6
3  1.928077  0.612458  4.790863  2.781058           5
4  2.380992  0.221706  1.164632  5.522668          20
5  3.509404  0.721268  5.290972  5.092809           2
6  2.475350  0.566563  2.638268  3.118178          14
7  0.547996  0.856024  2.092802  7.740978          11
8  3.073571  0.109006  6.406744  1.81

In [4]:
# EVALUATE BATCH
# Make inputs / outputs
batch = pd.DataFrame(initial_batch)
evaluation_keys = ["instance_name", "seed", "score"]
batch_evaluations = {idx:pd.DataFrame(columns=evaluation_keys) for idx, *_ in batch.iterrows()}

X = batch.values

# shouldn't do it like this, I need to evaluate configurations in multiple instances and seeds.
y = evotoon.evaluate_batch(batch, batch_evaluations, evotoon.execute_AntKnapsack, **function_kwargs)
y

{0:                                     instance_name  seed     score
 0  ./AntKnapsackClean-master/instances/weing1.txt   283 -1.415650
 1  ./AntKnapsackClean-master/instances/weing2.txt   284 -0.664716
 2  ./AntKnapsackClean-master/instances/weing3.txt   285 -0.052259
 3  ./AntKnapsackClean-master/instances/weing4.txt   286 -0.000000
 4  ./AntKnapsackClean-master/instances/weing7.txt   287 -0.458079,
 1:                                     instance_name  seed     score
 0  ./AntKnapsackClean-master/instances/weing1.txt   283 -2.275660
 1  ./AntKnapsackClean-master/instances/weing2.txt   284 -1.772580
 2  ./AntKnapsackClean-master/instances/weing3.txt   285 -3.364440
 3  ./AntKnapsackClean-master/instances/weing4.txt   286 -0.000000
 4  ./AntKnapsackClean-master/instances/weing7.txt   287 -0.108084,
 2:                                     instance_name  seed     score
 0  ./AntKnapsackClean-master/instances/weing1.txt   283 -0.353912
 1  ./AntKnapsackClean-master/instances/weing2.txt 

In [45]:
cols = ["instance_name"] + [f"conf_{idx}" for idx in batch_evaluations]

df = pd.DataFrame(columns=cols)
for instance_name in instance_list:
	evaluation_dict = {f"conf_{idx}": batch_evaluations[idx].loc[batch_evaluations[idx]["instance_name"] == instance_name]["score"].mean() for idx in batch_evaluations}
	new_row = {**{"instance_name":instance_name}, **evaluation_dict}
	df = df.append(new_row, ignore_index=True)
#df = pd.concat([df["instance_name"], df.rank(axis=1, numeric_only=True)], axis=1, join="inner")

df = df.rank(axis=1, numeric_only=True)
df


,conf_0,conf_1,conf_2,conf_3,conf_4,conf_5,conf_6,conf_7,conf_8,conf_9
0,3.0,2.0,9.0,4.0,9.0,6.0,6.0,1.0,9.0,6.0
1,4.0,2.0,9.5,7.0,7.0,3.0,5.0,1.0,9.5,7.0
2,7.5,2.0,4.5,9.5,9.5,4.5,6.0,1.0,3.0,7.5
3,6.0,6.0,6.0,6.0,6.0,6.0,6.0,1.0,6.0,6.0
4,3.0,6.0,9.0,7.0,4.0,1.0,10.0,2.0,8.0,5.0


In [65]:
from scipy.stats import friedmanchisquare
# compare samples
stat, p = friedmanchisquare(df["conf_0"], df["conf_1"], df["conf_2"],df["conf_3"], df["conf_4"], df["conf_5"],df["conf_6"], df["conf_7"], df["conf_8"])
print('Statistics=%.3f, p=%.3f' % (stat, p))
# interpret
alpha = 0.05
if p > alpha:
	print('Same distributions (fail to reject H0)')
else:
	print('Different distributions (reject H0)')

Statistics=23.085, p=0.003
Different distributions (reject H0)


In [67]:
import scipy.stats as ss
import scikit_posthocs as sp

sp.posthoc_conover(df, val_col='value', group_col='tag', p_adjust = 'holm')

ValueError: Specify correct column names using `group_col` and `val_col` args

In [54]:
data1 = list(range(100)) 


In [40]:
# EVALUATE BATCH
# Make inputs / outputs
batch = pd.DataFrame(initial_batch)
evaluation_keys = ["instance_name", "seed", "score"]
batch_evaluations = {idx:pd.DataFrame(columns=evaluation_keys) for idx, *_ in batch.iterrows()}

X = batch.values

# shouldn't do it like this, I need to evaluate configurations in multiple instances and seeds.
y = evotoon.evaluate_batch(initial_batch, evotoon.execute_AntKnapsack, **function_kwargs)


# Create model (this should consider the instance label)
model = evotoon.create_model(X)
history = model.fit(X, y, epochs=50, verbose=0, validation_split = 0.2)

for i in range(10):
	# Generate configurations from model
	generated_X = evotoon.generate_configurations(X, float_params) #just use LHS for the moment

	# Evaluate current batch
	A = model.predict(generated_X)

	batch = [{a : b for a,b in zip(initial_batch[0].keys(), conf)} for conf in X]
	B = evotoon.evaluate_batch(batch, evotoon.execute_AntKnapsack, **function_kwargs)
	B = B.reshape(B.size, 1)

	# Select configurations for next step
	partition_A = np.argpartition(A[:, -1], -poblation_size//2)[-poblation_size//2:]
	X_generated = generated_X[partition_A]
	y_generated = A[partition_A]

	partition_B = np.argpartition(B[:, -1], - (poblation_size - poblation_size//2))[-(poblation_size-poblation_size//2):] 
	X_new = X[partition_B]
	y_new = B[partition_B]

	X = np.concatenate((X_generated, X_new), axis=0)
	y = np.concatenate((y_generated, y_new), axis=0)

	# Update model
	history = model.fit(X_new, y_new, epochs=50, verbose=0, validation_split = 0.2)

	print(f"MAX (percentual difference with optimal) found in iteration {i}:", np.amax(B))



MAX (percentual difference with optimal) found in iteration 0: -0.11945832399999996
MAX (percentual difference with optimal) found in iteration 1: -0.11945832399999996
MAX (percentual difference with optimal) found in iteration 2: -0.11833728799999996
MAX (percentual difference with optimal) found in iteration 3: -0.11833728799999996
MAX (percentual difference with optimal) found in iteration 4: -0.11833728799999996
MAX (percentual difference with optimal) found in iteration 5: -0.11110703199999997
MAX (percentual difference with optimal) found in iteration 6: -0.11110703199999997
MAX (percentual difference with optimal) found in iteration 7: -0.11110703199999997
MAX (percentual difference with optimal) found in iteration 8: -0.11110703199999997
MAX (percentual difference with optimal) found in iteration 9: -0.11110703199999997


In [5]:
# Show best configurations found
print("Best configurations found:")
final_partition = np.argpartition(y_new[:, -1], - (poblation_size - poblation_size//2))[-(poblation_size-poblation_size//2):]
final_x = pd.DataFrame([{a : b for a,b in zip(initial_batch[0].keys(), conf)} for conf in X_new[final_partition]])
final_y = y_new[final_partition]
final_x["OPTIMAL DIFF"] = final_y * -1
final_x

Best configurations found:


,tau_max,rho,alpha,beta,total_ants,OPTIMAL DIFF
0,0.804733,0.340779,6.098550,6.243100,20.0,0.089642
1,0.354931,0.340779,6.098550,6.243100,20.0,0.083161
2,0.524733,0.300696,6.976714,4.906164,18.0,0.071933
3,0.257771,0.300696,1.610770,4.906164,18.0,0.071933
4,0.257771,0.445098,3.719741,4.906164,18.0,0.071933


In [1]:
import scipy.stats as ss
import scikit_posthocs as sp

import pandas as pd
import numpy as np

In [4]:
df = pd.DataFrame(columns=["value","tag"])

arr = np.arange(0, 10)
names = ["config_1" for i in range(len(arr))]

zipi = zip(arr,names)
df = df.append(pd.DataFrame(zipi, columns=df.columns), ignore_index=True)



arr2 = np.arange(10, 20)
names2 = ["config_2" for i in range(len(arr))]

zipi2 = zip(arr2,names2)
df = df.append(pd.DataFrame(zipi2, columns=df.columns), ignore_index=True)

arr3 = np.arange(50, 60)
names3 = ["config_3" for i in range(len(arr-2))]

zipi3 = zip(arr3,names3)
df = df.append(pd.DataFrame(zipi3, columns=df.columns), ignore_index=True)

df


,value,tag
0,0,config_1
1,1,config_1
2,2,config_1
3,3,config_1
4,4,config_1
5,5,config_1
6,6,config_1
7,7,config_1
8,8,config_1
9,9,config_1


In [5]:
sp.posthoc_conover(df, val_col='value', group_col='tag', p_adjust = 'holm')

,config_1,config_2,config_3
config_1,1.000000e+00,1.210492e-07,5.599870e-14
config_2,1.210492e-07,1.000000e+00,1.210492e-07
config_3,5.599870e-14,1.210492e-07,1.000000e+00
